In [3]:
import pickle
import os
import pandas as pd
from string import punctuation
import random

from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet, stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Utility**

In [4]:
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()
eng_stop = stopwords.words('english')

### **Preprocessing**

In [5]:
def AlterTag (tag: str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    return 'n'

def Preprocessing(docx: str):
    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in eng_stop]
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]

    tagged = pos_tag(tokens)

    tokens = [lemmatizer.lemmatize(tok, AlterTag(tag)) for tok, tag in tagged]
    tokens = [stemmer.stem(tok) for tok in tokens]
    return tokens

### **Training**

In [6]:
def Training():
    print('Initiate Training...')
    print('')
    data = pd.read_csv('./financial_dataset.csv')
    X = data['Statement']
    Y = data['Sentiment']

    # Feature
    feats = []

    for text, label in zip(X, Y):
        clean = Preprocessing(text)
        ft = dict(FreqDist(clean))

        # Atau Pakai ini:
        # Preferensi pribadi sih
        # ft = {word: True for word in clean}

        feats.append((ft, label))
    
    random.shuffle(feats)

    # Training
    print('Start Training...')
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]

    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    print('Model Trained')
    print(f'Accuracy: {acc}')
    print('')

    # Info
    print('Top 5 Most Informative Features')
    model.show_most_informative_features()
    print('')

    # Save
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    print('Model Saved')
    print('')

    return model


def Load():
    model = None
    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        print('No Model Detected')
        model = Training()
        return model

### **Support Function**

In [16]:
def Write_State ():
    docx = ''

    while True:
        print('Enter your Statement: ')
        docx = input('>> ')

        if len(docx.split()) < 2:
            print('Please Enter at least 2 words')
        else:
            return docx
        
def Analyze_State (model, docx: str):
    if len(docx.split()) < 2:
        print('Please Enter your Statement First')
        return None
    
    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]
    # tokens = [tok for tok in tokens if tok not in eng_stop]

    # POS Tag
    tagged = pos_tag(tokens)

    print('POS Tag:')
    for tok, tag in tagged:
        print(f'-> {tok}: {tag}')
    print('')

    # Syno Anto
    for word in tokens:
        print(f'Word: {word}')
        print('=' * (6 + len(word)))
        synsetx = wordnet.synsets(word)
        synonyms = []
        antonyms = []

        for sys in synsetx:
            for lemma in sys.lemmas():
                synonyms.append(lemma.name())
                for anton in lemma.antonyms():
                    antonyms.append(anton.name())

        synonyms = list(set(synonyms))
        antonyms = list(set(antonyms))
        
        print('Synonyms:')
        if len(synonyms) == 0:
            print('No Synonyms Detected')
        else:
            for w in synonyms:
                print(f'(+) {w}')
        print('')

        print('Antonyms:')
        if len(antonyms) == 0:
            print('No Antonyms Detected')
        else:
            for w in antonyms:
                print(f'(-) {w}')
        print('')


    # Prediction
    clean = Preprocessing(docx)
    feats = {word: True for word in clean}

    pred = model.classify(feats)
    print(f'The Statement is classified as {pred}')



### **Menu**

In [8]:
def Menu():
    docx = ''
    model = Load()

    while True:
        print('1. Write Your Statement')
        print('2. Analyze Your Statement')
        print('3. End Session')
        cc = input('>>')

        print('')
        if cc == '1':
            docx = Write_State()
        elif cc == '2':
            Analyze_State(model, docx)
        elif cc == '3':
            print('Thank You for using our app :)')
            break
        else:
            print('Invalid Input')
        

        print('')

In [17]:
Menu()

1. Write Your Statement
2. Analyze Your Statement
3. End Session

Enter your Statement: 

1. Write Your Statement
2. Analyze Your Statement
3. End Session

POS Tag:
-> please: VB
-> enter: NN
-> at: IN
-> least: JJS
-> words: NNS

Word: please
Synonyms:
(+) delight
(+) please

Antonyms:
(-) displease

Word: enter
Synonyms:
(+) introduce
(+) figure
(+) insert
(+) enter
(+) record
(+) enrol
(+) move_into
(+) come_in
(+) put_down
(+) enroll
(+) get_in
(+) accede
(+) infix
(+) go_into
(+) get_into
(+) embark
(+) inscribe
(+) participate
(+) go_in
(+) recruit

Antonyms:
(-) drop_out
(-) exit

Word: at
Synonyms:
(+) atomic_number_85
(+) at
(+) astatine
(+) At

Antonyms:
No Antonyms Detected

Word: least
Synonyms:
(+) least
(+) to_the_lowest_degree

Antonyms:
(-) most

Word: words
Synonyms:
(+) language
(+) wrangle
(+) Christian_Bible
(+) Scripture
(+) watchword
(+) Good_Book
(+) formulate
(+) phrase
(+) countersign
(+) Holy_Writ
(+) lyric
(+) Book
(+) give-and-take
(+) words
(+) Son
(+) pass